# TP1 — Data Preprocessing & Train a Model (Weather Prediction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session3/tp1.ipynb)

## Objective

Explore real-world weather data, handle quality issues (missing values, outliers), engineer features, and **train a model** that predicts Paris weather 6 hours ahead.

The model you save at the end of this TP will be used in **TP2** to submit to the live competition.

### Roadmap

| Section | What you do |
|---------|-------------|
| 1. Load & Explore | Understand the dataset structure |
| 2. Visualize | See temporal patterns and what "+6h prediction" means |
| 3. Missing Values | Detect, visualize, and handle missing data |
| 4. Outliers | Detect and handle extreme values |
| 5. Feature Engineering | Create useful features for prediction |
| 6. Model Training | Baseline → Linear Regression → Ensemble methods |
| 7. Save Model | Export your best model for TP2 |

---
## 1. Setup

In [ ]:
!pip install scikit-learn pandas matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

---
## 2. Load & Explore

The dataset contains hourly weather data for **20 European cities** over the full year 2025. Each row is one city at one hour.

Columns: `timestamp`, `city_name`, `country_code`, `latitude`, `longitude`, `temperature`, `rain`, `wind_speed`, `wind_direction`, `humidity`, `clouds`, `snow`

### Exercise 1.1 — Load the CSV

Load `weather_paris_20cities.csv`, parse timestamps, and display shape, first rows, and dtypes.

*Hint:* `pd.read_csv(..., parse_dates=['timestamp'])`, `df.shape`, `df.head()`, `df.dtypes`

In [ ]:
# YOUR CODE HERE

You should see ~184K rows (20 cities × 8760 hours) and 12 columns.

### Exercise 1.2 — Summary statistics per city

Group by city and compute mean temperature, mean wind_speed, and mean rain. Which city is warmest? Coldest?

*Hint:* `df.groupby('city_name')[['temperature', 'wind_speed', 'rain']].mean()`

In [ ]:
# YOUR CODE HERE

### Exercise 1.3 — Check missing values

Count missing values per column, and per city. Are missing values evenly distributed?

*Hint:* `df.isnull().sum()`, `df.groupby('city_name').apply(lambda g: g.isnull().sum())`

**Question:** Does Paris have any missing values?

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*

---
## 3. Visualize & Understand the Objective

Before cleaning the data, let's visualize it to understand temporal patterns and what we're trying to predict.

### Exercise 2.1 — Plot Paris temperature over time

Filter to Paris and plot temperature over the full year. What patterns do you see?

*Hint:* `df[df['city_name']=='Paris'].plot(x='timestamp', y='temperature')`

In [ ]:
# YOUR CODE HERE

### Exercise 2.2 — Plot multiple cities — spatial correlation

On the same plot, show January temperature for Paris, London, Barcelona, and Munich. Do nearby cities have correlated weather?

*Hint:* Filter to January, loop over cities, `plt.plot(...)` for each.

In [ ]:
# YOUR CODE HERE

### Exercise 2.3 — Visualize what "+6h prediction" means

For Paris, create a column `temperature_t6` = temperature shifted by -6 hours (the value 6 hours in the future). Scatter plot `temperature` (now) vs `temperature_t6` (+6h). How correlated are they?

*Hint:* `paris['temperature_t6'] = paris['temperature'].shift(-6)`, `plt.scatter(...)`

**Question:** If you just predicted "temperature stays the same", how wrong would you be on average?

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*

---
## 4. Handle Missing Values

Real-world data is messy. The dataset has ~2% missing values in non-Paris cities. We need to handle them before training.

### Exercise 3.1 — Visualize missing patterns

Create a heatmap showing the fraction of missing values per city and per feature.

*Hint:* `df.groupby('city_name')[numeric_cols].apply(lambda g: g.isnull().mean())`, `sns.heatmap(...)`

In [ ]:
# YOUR CODE HERE

### Exercise 3.2 — Try strategies: drop, forward fill, interpolation

Pick one city with missing values (e.g., London). For its temperature column:
1. **Drop** rows with missing values — how many rows lost?
2. **Forward fill** (`ffill`) — what assumption does this make?
3. **Linear interpolation** — `interpolate(method='linear')`

Plot all three on the same chart for a short time window (e.g., one week in January).

*Hint:* Work on a copy each time. Use `.loc[start:end]` to zoom in.

In [ ]:
# YOUR CODE HERE

### Exercise 3.3 — Apply your chosen strategy

Choose the best strategy and apply it to the full dataset. Verify there are no more missing values.

**Question:** Which strategy did you choose and why?

*Hint:* Group by city, then interpolate within each group to avoid cross-city interpolation.

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*

---
## 5. Detect & Handle Outliers

The dataset also contains ~0.5% outliers (impossible or extreme values) in non-Paris cities.

### Exercise 4.1 — Box plots — spot outliers

Create box plots for `temperature`, `humidity`, `wind_speed`, and `rain` across all cities. Do you see extreme values?

*Hint:* `fig, axes = plt.subplots(2, 2)`, `sns.boxplot(data=df, y=col, ax=axes[i])`

In [ ]:
# YOUR CODE HERE

### Exercise 4.2 — IQR-based outlier detection

For each numeric column, compute Q1, Q3, and IQR. Flag values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` as outliers. How many outliers per column?

*Hint:* `Q1 = df[col].quantile(0.25)`, `IQR = Q3 - Q1`, use boolean masks.

In [ ]:
# YOUR CODE HERE

### Exercise 4.3 — Clip outliers and compare

Replace outlier values by clipping to the IQR bounds. Compare distributions before/after with histograms or box plots.

*Hint:* `df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)`

In [ ]:
# YOUR CODE HERE

---
## 6. Feature Engineering

Raw data rarely works well for ML. We need to create features that help the model learn patterns.

### Exercise 5.1 — Drop unnecessary columns

Drop `country_code` (redundant with city_name). Discuss: `latitude` and `longitude` could encode spatial relationships, but since we'll pivot cities into separate columns, we can drop them too for now.

*Hint:* `df = df.drop(columns=['country_code', 'latitude', 'longitude'])`

In [ ]:
# YOUR CODE HERE

### Exercise 5.2 — Create the target and pivot cities

Our goal: predict Paris temperature, wind_speed, and rain at T+6h.

Steps:
1. For Paris, create target columns: shift `temperature`, `wind_speed`, `rain` by -6 to get T+6h values
2. Pivot the data so each row = one timestamp, with columns like `Paris_temperature`, `London_temperature`, etc.
3. Add Paris lag features: T-1, T-2, ..., T-6 for temperature, wind_speed, rain

*Hint:*
- `pivot_table(index='timestamp', columns='city_name', values=[...])` 
- Flatten MultiIndex columns: `df.columns = ['_'.join(col) for col in df.columns]`
- `df['Paris_temperature_lag1'] = df['Paris_temperature'].shift(1)`

In [ ]:
# YOUR CODE HERE

### Exercise 5.3 — Cyclical time features

The hour of day is cyclical: hour 23 and hour 0 are close, but numerically far apart. Encoding hour as a raw integer misleads the model.

Instead, encode time as **sin/cos** pairs:
```python
sin_hour = sin(2π × hour / 24)
cos_hour = cos(2π × hour / 24)
```

This preserves the circular structure: hour 23 and hour 0 become neighbors in the sin/cos space.

*Hint:* `df['sin_hour'] = np.sin(2 * np.pi * df.index.hour / 24)`

In [ ]:
# YOUR CODE HERE

---
## 7. Baseline & Model Training

Now we have clean data with engineered features. Let's train models to predict Paris weather at T+6h.

**Target:** `[temperature_t6, wind_speed_t6, rain_t6]` for Paris

We'll train a separate model for each target (or a multi-output model) and compare with a naive baseline.

### Exercise 6.1 — Prepare train/test split, scale features, and select features

1. Define your feature columns (all city weather + lags + time features) and target columns (Paris T+6h)
2. Drop rows with NaN (from shifting)
3. Split into train/test — use a **temporal split** (first 80% for training, last 20% for testing) since this is time series data
4. **Scale features** with `StandardScaler` — fit on train only, transform both. This helps Linear Regression converge and makes feature importances comparable.
5. **Feature selection** — compute correlation between each feature and the targets. Keep only features with |correlation| above a threshold (e.g., 0.05). This removes noise features that hurt generalization.

**Why temporal split?** Random splits would leak future information into training. In a real competition, your model only sees past data.

*Hint:*
- `split_idx = int(len(df) * 0.8)`, `train = df.iloc[:split_idx]`, `test = df.iloc[split_idx:]`
- `scaler = StandardScaler()`, `X_train_scaled = scaler.fit_transform(X_train)`, `X_test_scaled = scaler.transform(X_test)`
- `correlations = X_train.corrwith(y_train['target_temperature']).abs()`

In [ ]:
# YOUR CODE HERE

### Exercise 6.2 — Naive baseline: last known Paris value

The simplest prediction: assume Paris weather in 6 hours = Paris weather right now. Compute MAE for each target.

*Hint:* Predictions = current Paris values, compare with actual T+6h values.

In [ ]:
# YOUR CODE HERE

### Exercise 6.3 — Linear Regression

Train a `LinearRegression` on your features. Compute MAE for each target. Does it beat the baseline?

*Hint:* You can train one model per target or use `LinearRegression` which natively supports multi-output.

In [ ]:
# YOUR CODE HERE

### Exercise 6.4 — Random Forest / Gradient Boosting

Train a `RandomForestRegressor` and/or `GradientBoostingRegressor`. Compare MAE with previous approaches.

*Hint:* `GradientBoostingRegressor` does not support multi-output natively — use `sklearn.multioutput.MultiOutputRegressor` to wrap it, or train 3 separate models.

In [ ]:
# YOUR CODE HERE

### Exercise 6.5 — Compare all approaches

Create a comparison table and bar chart showing MAE for each approach and each target.

Also compute the **competition metric** — Negated Normalized MAE:
```
score = -mean(|pred - true| / std)
```
where `std_temperature = 7.49`, `std_wind_speed = 5.05`, `std_rain = 0.40`.

*Hint:* `pd.DataFrame(results).set_index('model')`, `df.plot(kind='bar')`

In [ ]:
# YOUR CODE HERE

---
## 8. Save Your Best Model

Save your best model so we can load it in TP2 and submit it to the competition.

### Exercise 7.1 — Save with joblib

Save your best model, the scaler, and the selected feature list to files. We'll load them in TP2.

*Hint:* `joblib.dump(model, 'model.pkl')`, `joblib.dump(scaler, 'scaler.pkl')`, `joblib.dump(selected_features, 'features.pkl')`

**Important:** You need to save the scaler and feature list because the Agent must apply the exact same transformation at prediction time.

In [ ]:
# YOUR CODE HERE

---
## 9. Summary

### Key Takeaways

1. **Real data is messy** — missing values and outliers must be handled before training
2. **Visualization first** — always look at your data before applying transformations
3. **Cyclical encoding** — use sin/cos for periodic features (hour of day), not raw integers
4. **Feature selection** — remove low-correlation features to reduce noise
5. **Scaling** — StandardScaler helps Linear Regression and makes features comparable
6. **Temporal splits** — for time series, never use random train/test splits
7. **Always compare to a baseline** — if your model can't beat "predict last known value", something is wrong

### What you built

| Step | What you did |
|------|-------------|
| **Explore** | Loaded 20-city weather data, checked structure and quality |
| **Visualize** | Understood temporal patterns and T+6h prediction task |
| **Clean** | Handled ~2% missing values + ~0.5% outliers |
| **Engineer** | Created lag, cross-city, and cyclical time features |
| **Select** | Removed low-correlation features |
| **Scale** | Applied StandardScaler (fit on train only) |
| **Train** | Compared baseline, linear, and ensemble models |
| **Save** | Exported model + scaler + feature list |

### Next: TP2

In TP2, you'll wrap this model into an `Agent` class, test it locally (reproducing the platform's evaluation), and submit it to the live competition. You'll have ~1 month by team to iterate and climb the leaderboard.